# Feature Relationship Explorer

Quick visual read of how the current baseline and candidate Gold features relate to each other and to the `is_fraud` label.

**Current three-feature baseline:** `prior_amount_zscore`, `amount_sum_last_1h`, `is_night_transaction`

**Candidates under review:** `tx_count_last_1h`, `merchant_prior_fraud_rate`, `hour_of_day`

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root.parent != project_root:
    project_root = project_root.parent

src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from fraud_lens.ingest import load_sparkov_config

sparkov_cfg = load_sparkov_config().get("sparkov", {})
gold_path = (project_root / sparkov_cfg.get("gold_path", "data/benchmark/gold_sparkov")).resolve()

spark_builder = SparkSession.builder.appName("FraudLens-Feature-Relationships")
for key, value in sparkov_cfg.get("spark_runtime", {}).items():
    spark_builder = spark_builder.config(key, str(value))
spark = spark_builder.getOrCreate()

print(f"Gold path: {gold_path}")

In [ ]:
FEATURES = [
    "prior_amount_zscore",
    "amount_sum_last_1h",
    "is_night_transaction",
    "hour_of_day",
    "tx_count_last_1h",
    "merchant_prior_fraud_rate",
]

gold_df = spark.read.parquet(str(gold_path))

sample_df = (
    gold_df.select("is_fraud", *FEATURES)
    .where(F.col("prior_amount_zscore").isNotNull())
    .withColumn(
        "label",
        F.when(F.col("is_fraud") == 1, F.lit("fraud")).otherwise(F.lit("non_fraud")),
    )
    .sampleBy("label", fractions={"fraud": 1.0, "non_fraud": 0.01}, seed=42)
)

pdf = sample_df.toPandas()
pdf["is_night_transaction"] = pdf["is_night_transaction"].astype(int)

fraud = pdf[pdf["label"] == "fraud"]
legit = pdf[pdf["label"] == "non_fraud"]
print(f"Sampled rows — fraud: {len(fraud):,}, non_fraud: {len(legit):,}")

## 1 — Correlation Heatmap

Pearson correlations among the candidate features plus the fraud label. High inter-feature correlation means the features carry redundant signal; low correlation with `is_fraud` means weak standalone predictive value.

In [ ]:
corr_cols = FEATURES + ["is_fraud"]
corr_df = pdf[corr_cols].apply(pd.to_numeric, errors="coerce")
corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(corr_cols, fontsize=9)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        val = corr.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=8, color="white" if abs(val) > 0.5 else "black")

fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Feature Correlation Matrix (sampled Sparkov Gold)")
fig.tight_layout()
plt.show()

## 2 — Pairwise Scatter Matrix

Scatter matrix of the six candidate features, colored by fraud label. Fraud points are drawn on top for visibility. Axes are clipped at the 99th percentile of the combined sample to avoid compression from outliers.

In [ ]:
pair_features = [
    "prior_amount_zscore",
    "amount_sum_last_1h",
    "hour_of_day",
    "tx_count_last_1h",
    "merchant_prior_fraud_rate",
]

n = len(pair_features)
fig, axes = plt.subplots(n, n, figsize=(16, 16))
colors = {"non_fraud": "#4c78a8", "fraud": "#e45756"}

clip_hi = pdf[pair_features].quantile(0.99)
clip_lo = pdf[pair_features].quantile(0.01)

for i, fy in enumerate(pair_features):
    for j, fx in enumerate(pair_features):
        ax = axes[i][j]
        if i == j:
            for lbl, color in colors.items():
                subset = pdf[pdf["label"] == lbl][fy].clip(clip_lo[fy], clip_hi[fy])
                ax.hist(subset, bins=40, alpha=0.5, color=color, label=lbl, density=True)
            if i == 0:
                ax.legend(fontsize=7)
        else:
            ax.scatter(
                legit[fx].clip(clip_lo[fx], clip_hi[fx]),
                legit[fy].clip(clip_lo[fy], clip_hi[fy]),
                s=1, alpha=0.15, c=colors["non_fraud"], rasterized=True,
            )
            ax.scatter(
                fraud[fx].clip(clip_lo[fx], clip_hi[fx]),
                fraud[fy].clip(clip_lo[fy], clip_hi[fy]),
                s=4, alpha=0.5, c=colors["fraud"], rasterized=True,
            )
        if j == 0:
            ax.set_ylabel(fy, fontsize=8)
        if i == n - 1:
            ax.set_xlabel(fx, fontsize=8)
        ax.tick_params(labelsize=6)

fig.suptitle("Pairwise Feature Scatter (fraud = red, non-fraud = blue)", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 3 — Feature Distributions by Fraud Label

Side-by-side box plots for each candidate feature, split by fraud vs non-fraud. Boxes are clipped at the 1st–99th percentile to keep the view interpretable.

In [ ]:
box_features = [
    "prior_amount_zscore",
    "amount_sum_last_1h",
    "hour_of_day",
    "tx_count_last_1h",
    "merchant_prior_fraud_rate",
]

fig, axes = plt.subplots(1, len(box_features), figsize=(18, 5))

for idx, feat in enumerate(box_features):
    ax = axes[idx]
    lo = pdf[feat].quantile(0.01)
    hi = pdf[feat].quantile(0.99)
    data_l = legit[feat].clip(lo, hi).dropna()
    data_f = fraud[feat].clip(lo, hi).dropna()
    bp = ax.boxplot(
        [data_l, data_f],
        labels=["non-fraud", "fraud"],
        patch_artist=True,
        widths=0.6,
        showfliers=False,
    )
    bp["boxes"][0].set_facecolor("#4c78a8")
    bp["boxes"][1].set_facecolor("#e45756")
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=8)

fig.suptitle("Feature Distributions by Fraud Label (1st–99th pct)", fontsize=12)
fig.tight_layout()
plt.show()

## 4 — Baseline Feature Interactions

Focused scatter of the current three-feature logistic baseline. Each panel pairs two of the three features and colors points by fraud label. A separate panel shows the hour-of-day distribution stacked by fraud, which is the next candidate beyond `is_night_transaction`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
colors_map = {"non_fraud": "#4c78a8", "fraud": "#e45756"}

# Panel A: prior_amount_zscore vs amount_sum_last_1h
ax = axes[0, 0]
for lbl in ["non_fraud", "fraud"]:
    s = pdf[pdf["label"] == lbl]
    ax.scatter(
        s["prior_amount_zscore"].clip(-5, 20),
        np.log1p(s["amount_sum_last_1h"]),
        s=2 if lbl == "non_fraud" else 6,
        alpha=0.15 if lbl == "non_fraud" else 0.45,
        c=colors_map[lbl], label=lbl, rasterized=True,
    )
ax.set_xlabel("prior_amount_zscore (clipped [-5, 20])")
ax.set_ylabel("log1p(amount_sum_last_1h)")
ax.set_title("Amount Z-Score vs 1h Amount Sum")
ax.legend(fontsize=8, markerscale=3)

# Panel B: prior_amount_zscore vs is_night_transaction
ax = axes[0, 1]
for lbl in ["non_fraud", "fraud"]:
    s = pdf[pdf["label"] == lbl]
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(s))
    ax.scatter(
        s["is_night_transaction"].astype(float) + jitter,
        s["prior_amount_zscore"].clip(-5, 20),
        s=2 if lbl == "non_fraud" else 6,
        alpha=0.12 if lbl == "non_fraud" else 0.4,
        c=colors_map[lbl], label=lbl, rasterized=True,
    )
ax.set_xlabel("is_night_transaction (jittered)")
ax.set_ylabel("prior_amount_zscore (clipped [-5, 20])")
ax.set_title("Night Flag vs Amount Z-Score")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Day", "Night"])
ax.legend(fontsize=8, markerscale=3)

# Panel C: amount_sum_last_1h vs is_night_transaction
ax = axes[1, 0]
for lbl in ["non_fraud", "fraud"]:
    s = pdf[pdf["label"] == lbl]
    jitter = np.random.default_rng(1).uniform(-0.15, 0.15, len(s))
    ax.scatter(
        s["is_night_transaction"].astype(float) + jitter,
        np.log1p(s["amount_sum_last_1h"]),
        s=2 if lbl == "non_fraud" else 6,
        alpha=0.12 if lbl == "non_fraud" else 0.4,
        c=colors_map[lbl], label=lbl, rasterized=True,
    )
ax.set_xlabel("is_night_transaction (jittered)")
ax.set_ylabel("log1p(amount_sum_last_1h)")
ax.set_title("Night Flag vs 1h Amount Sum")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Day", "Night"])
ax.legend(fontsize=8, markerscale=3)

# Panel D: hour_of_day histogram stacked by label
ax = axes[1, 1]
bins = np.arange(-0.5, 24.5, 1)
ax.hist(
    [legit["hour_of_day"], fraud["hour_of_day"]],
    bins=bins, density=True, stacked=False, alpha=0.55,
    color=[colors_map["non_fraud"], colors_map["fraud"]],
    label=["non_fraud", "fraud"],
)
ax.axvspan(-0.5, 6, alpha=0.08, color="gray", label="night zone")
ax.axvspan(22, 23.5, alpha=0.08, color="gray")
ax.set_xlabel("hour_of_day")
ax.set_ylabel("density")
ax.set_title("Hour-of-Day Distribution by Label")
ax.legend(fontsize=8)

fig.suptitle("Current Baseline Feature Interactions", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 5 — Fraud Rate by Feature Bin

For each continuous feature, bin transactions and compute the empirical fraud rate per bin. This shows where in feature space the model's signal actually lives, independent of the class imbalance.

In [ ]:
bin_features = [
    ("prior_amount_zscore", (-3, 15), 30),
    ("amount_sum_last_1h", (0, 3000), 30),
    ("hour_of_day", (-0.5, 23.5), 24),
    ("tx_count_last_1h", (0.5, 10.5), 10),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for idx, (feat, (lo, hi), nbins) in enumerate(bin_features):
    ax = axes[idx // 2][idx % 2]
    subset = pdf[[feat, "is_fraud"]].dropna()
    subset = subset[(subset[feat] >= lo) & (subset[feat] <= hi)]
    subset["bin"] = pd.cut(subset[feat], bins=nbins)
    grouped = subset.groupby("bin", observed=True)["is_fraud"].agg(["mean", "count"])
    mids = [interval.mid for interval in grouped.index]
    ax.bar(mids, grouped["mean"], width=(hi - lo) / nbins * 0.85,
           color="#e45756", alpha=0.7, edgecolor="white")
    ax.set_xlabel(feat)
    ax.set_ylabel("fraud rate")
    ax.set_title(f"Fraud Rate by {feat}")
    ax2 = ax.twinx()
    ax2.plot(mids, grouped["count"], color="gray", alpha=0.5, marker=".", linewidth=1)
    ax2.set_ylabel("sample count", color="gray", fontsize=8)
    ax2.tick_params(axis="y", labelcolor="gray", labelsize=7)

fig.suptitle("Empirical Fraud Rate by Feature Bin (red bars = fraud rate, gray line = bin count)", fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
spark.stop()